# MotionBricks on Colab (T4)

[NVlabs/motionbricks](https://github.com/NVlabs/motionbricks) の SOMA モデルで、クリップから取った姿勢を順に置いてつなぐモーションを生成し、BVH と骨格の動画 (mp4) を書き出す。

GPU ランタイム (T4) で実行する。`/content/work/` に、このノートブックと同じディレクトリの `generate.py`、`render.py`、`scenarios/` を置いておく (`make upload` が置く)。

In [ ]:
%%bash
# MotionBricks を clone して依存をインストールする (2 回目以降はスキップされる)
set -eu
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
command -v git-lfs >/dev/null || (apt-get -qq update && apt-get -qq install -y git-lfs)
command -v uv >/dev/null || pip install -q uv
cd /content
[ -d motionbricks ] || git clone -q --depth 1 https://github.com/NVlabs/motionbricks.git
cd motionbricks
git lfs install --local >/dev/null
git lfs pull
uv sync -q
ls /content/work/scenarios

In [ ]:
%%bash
# scenarios/*.json ごとにモーションを生成する (初回は TensorRT プランのビルドに数分かかる)
set -eu
cd /content/motionbricks
for scenario in /content/work/scenarios/*.json; do
  name=$(basename "$scenario" .json)
  .venv/bin/python /content/work/generate.py --scenario "$scenario" --out "/content/work/out/$name" 2>&1 | grep -v '^Module '
done

In [ ]:
%%bash
# 骨格の動画を描く
set -eu
for npz in /content/work/out/*.npz; do
  python3 /content/work/render.py "$npz"
done
ls -l /content/work/out

In [ ]:
# @title 動画を表示する (Web UI 用)
from pathlib import Path
from IPython.display import Video, display

for mp4 in sorted(Path("/content/work/out").glob("*.mp4")):
    print(mp4.name)
    display(Video(str(mp4), embed=True, width=480))